In [ ]:
# %load_ext autoreload
# %autoreload 2
import pygsti as pg
import numpy as np
import scipy.linalg as la

from pygsti.baseobjs import qubitgraph as _qgraph
from pygsti.protocols import StandardGSTDesign
from pygsti.protocols.xfgst_edesign import make_xfgst_design
from pygsti.modelpacks import smq1Q_XYI
from pygsti.processors import QubitProcessorSpec
from pygsti.baseobjs import QubitSpace
from pygsti.circuits import Circuit
from pygsti.protocols import CircuitListsDesign


In [2]:
class ModelingHelpers:

    @staticmethod
    def make_2q_XYIECR_cptplnd():
        from pygsti.models import modelconstruction as pgmc
        u_ecr = 1/np.sqrt(2)*np.array([[0,0,1,1j],[0,0,1j,1],[1,-1j,0,0],[-1j,1,0,0]])
        gatenames = ['Gxpi2', 'Gypi2', 'Gii',  'Gecr']
        ps = QubitProcessorSpec(
            num_qubits=2,
            gate_names=gatenames,
            nonstd_gate_unitaries={'Gecr': u_ecr, 'Gii': np.eye(4)},
            availability={
                'Gxpi2': [(0,), (1,)],
                'Gypi2': [(0,), (1,)],
                'Gecr' : [(0, 1), (1, 0)],
                'Gii'  : [(0, 1)]
            }
        )
        tm2 = pgmc.create_explicit_model(ps)
        tm2.convert_members_inplace('CPTPLND')
        return tm2

    @staticmethod
    def make_2q_edesign(target_model2Q, mml, find_fids_verbosity=0, germ_sel_verbosity=0):
        from pygsti.algorithms import fiducialselection as fidsel
        from pygsti.algorithms import germselection as germsel
        prep_fiducials2Q, meas_fiducials2Q = fidsel.find_fiducials(
            target_model2Q, algorithm='greedy', candidate_fid_counts=4, verbosity=find_fids_verbosity
        )
        germs2Q = germsel.find_germs(
            target_model2Q, randomize=False, algorithm='greedy', assume_real=True, float_type=np.double,
            candidate_germ_counts={5: 'all upto'}, mode='compactEVD', verbosity=germ_sel_verbosity
        )
        temp = [c for c in germs2Q]
        germs2Q = []
        for c in temp:
            new_c = c.copy(editable=True)
            new_c.line_labels = (0,1)
            new_c.done_editing()
            germs2Q.append(new_c)
        Ls = [2**k for k in range(round(np.log2(mml)+1))]
        from pygsti.algorithms.fiducialpairreduction import find_sufficient_fiducial_pairs_per_germ
        fidpairs = find_sufficient_fiducial_pairs_per_germ(target_model2Q, prep_fiducials2Q, meas_fiducials2Q, germs2Q)
        edesign2Q = StandardGSTDesign(target_model2Q, prep_fiducials2Q, meas_fiducials2Q, germs2Q, Ls, fiducial_pairs=fidpairs)
        return edesign2Q

    @staticmethod
    def make_nq_spam(num_qubits, sectors=('H',)):
        from pygsti.modelmembers.states import ComposedState, ComputationalBasisState
        from pygsti.modelmembers.povms import ComposedPOVM
        from pygsti.modelmembers.operations import LindbladErrorgen, ExpErrorgenOp
        from pygsti.baseobjs.errorgenbasis import CompleteElementaryErrorgenBasis
        state_space = QubitSpace(num_qubits)
        max_weights = {'H':1, 'S':1, 'C':1, 'A':1}
        egbn_H_only = CompleteElementaryErrorgenBasis('PP', state_space, sectors, max_weights)

        rho_errgen_rates = {ell: 0.0 for ell in egbn_H_only.labels}
        rho_lindblad = LindbladErrorgen.from_elementary_errorgens(rho_errgen_rates, state_space=state_space, evotype='densitymx')
        rho_errorgen = ExpErrorgenOp(rho_lindblad)
        rho_ideal    = ComputationalBasisState([0]*num_qubits)
        rho          = ComposedState(rho_ideal, rho_errorgen)

        M_errgen_rates = {ell: 0.0 for ell in egbn_H_only.labels}
        M_lindblad = LindbladErrorgen.from_elementary_errorgens(M_errgen_rates, state_space=state_space, evotype='densitymx')
        M_errorgen = ExpErrorgenOp(M_lindblad)
        M = ComposedPOVM(M_errorgen)
        return rho, M

    @staticmethod
    def make_nq_XYIECR_target_model(num_qubits, spam_sectors=('H',), gate_sectors=('H','S'), **create_xfmdl_kwargs):
        from pygsti.models import modelconstruction as pgmc
        from pygsti.baseobjs.errorgenbasis import CompleteElementaryErrorgenBasis
        # Step 1. Define the ideal model
        ps_geometry = _qgraph.QubitGraph.common_graph(
            num_qubits, geometry='line', directed=True, all_directions=True,
            qubit_labels=tuple(range(num_qubits))
        )
        u_ecr = 1/np.sqrt(2)*np.array([[0,0,1,1j],[0,0,1j,1],[1,-1j,0,0],[-1j,1,0,0]])
        gatenames = ['Gxpi2', 'Gypi2', 'Gi', 'Gii',  'Gecr']
        ps = QubitProcessorSpec(
            num_qubits=num_qubits, gate_names=gatenames,
            nonstd_gate_unitaries={'Gecr': u_ecr, 'Gii': np.eye(4)}, geometry=ps_geometry
        )
        # Step 2. build ingredients for setting the model parameterization
        gateerrs = dict()
        egb1 = CompleteElementaryErrorgenBasis('PP', QubitSpace(1), gate_sectors, default_label_type='local')
        for gn in gatenames[:-1]:
            gateerrs[gn] = {ell: 0 for ell in egb1.labels}
        egb2 = CompleteElementaryErrorgenBasis('PP', QubitSpace(2), gate_sectors, default_label_type='local')
        gateerrs['Gecr'] = {ell: 0 for ell in egb2.labels}
        gateerrs['Gii'] = gateerrs['Gecr']

        # Step 3. Create a proper "Model" object, capable of simulation.
        create_xfmdl_kwargs.setdefault('independent_gates', True)
        tmn = pgmc.create_crosstalk_free_model(ps, lindblad_error_coeffs=gateerrs, **create_xfmdl_kwargs)
        rho, M = ModelingHelpers.make_nq_spam(num_qubits, spam_sectors)
        tmn.prep_blks['layers']['rho0'] = rho
        tmn.povm_blks['layers']['Mdefault'] = M
        if create_xfmdl_kwargs['independent_gates']:
            # For every i >= 1, there's a redundant parameterized two-qubit idle gate between
            # qubits (i, i-1). We'll make this a view of the idle between qubits (i-1, i).
            for i in range(1, num_qubits):
                del tmn.operation_blks['gates'][('Gii', i, i-1)]
                tmn.operation_blks['layers'][('Gii', i, i-1)] = tmn.operation_blks['layers'][('Gii', i-1, i)]
        tmn._rebuild_paramvec()
        return tmn

    @staticmethod
    def make_nq_XYIECR_pspec(num_qubits, **create_xfmdl_kwargs):
        tm = ModelingHelpers.make_nq_XYIECR_target_model(num_qubits, **create_xfmdl_kwargs)
        # ^ The processor spec is unaffected by the model parameterization, so we don't 
        #   bother taking in sectors here.
        ps = tm.create_processor_spec()
        return ps
    
    @staticmethod
    def make_nq_XYIECR_edesign(num_qubits: int, maxmaxlen: int):
        ed_1q = smq1Q_XYI.create_gst_experiment_design(maxmaxlen)  # type: ignore
        tm_2q = ModelingHelpers.make_2q_XYIECR_cptplnd()
        ed_2q = ModelingHelpers.make_2q_edesign(tm_2q, maxmaxlen)
        ps_nq = ModelingHelpers.make_nq_XYIECR_pspec(num_qubits)
        edesign  = make_xfgst_design(ps_nq, ed_1q, ed_2q, seed=0)
        return edesign


In [5]:
num_qubits = 3
edesign  = ModelingHelpers.make_nq_XYIECR_edesign(num_qubits, maxmaxlen=8)
allcircuits = [c for c in edesign.all_circuits_needing_data]
print(len(allcircuits))


/Users/rjmurr/Documents/pg-xfgst/repo/pygsti/forwardsims/mapforwardsim.py:745: ForwardSimulatorSuitabilityWarning: 
            Generating dense process matrix representations of circuits or gates
            can be inefficient and should be avoided for the purposes of forward
            simulation/calculation of circuit outcome probability distributions
            when using the MapForwardSimulator. This operator is small enough
            that it could be computed with MatrixForwardSimulator instead.
            
  _warnings.warn(msg, ForwardSimulatorSuitabilityWarning)


9656
